## 🛠️ 환경 설정 (처음 실행 시)

이 프로젝트는 `uv`로 환경을 관리합니다.

### 1단계: 부트스트랩 (최초 1회)

```bash
# uv 설치 + Python 3.14 + 모든 의존성 설치
scripts/bootstrap.sh
```

### 2단계: Jupyter 실행

```bash
uv run jupyter notebook
# 또는
uv run jupyter lab
```

> 위 명령으로 실행하면 패키지가 모두 준비된 상태로 노트북이 열립니다.
> 아래 설치 확인 셀을 실행해 정상 설치 여부를 확인하세요.

---

### GPU (CUDA) 사용 시

`uv sync`로 설치되는 torch는 CPU 전용입니다.
GPU를 사용하려면 [pytorch.org/get-started/locally](https://pytorch.org/get-started/locally/)에서
본인 CUDA 버전에 맞는 명령어를 확인하여 별도 설치하세요.

```bash
# 예시 (CUDA 12.x)
uv run pip install torch --index-url https://download.pytorch.org/whl/cu121
```

In [2]:
# 패키지 설치 확인
import importlib, subprocess, sys

required = ['torch', 'neuralop', 'plotly', 'numpy']
missing  = [pkg for pkg in required if importlib.util.find_spec(pkg) is None]

if missing:
    print(f'⚠️  누락된 패키지: {missing}')
    print('위의 설치 안내를 따라 패키지를 설치하세요.')
else:
    import torch, plotly, numpy
    print(f'✅ torch   : {torch.__version__}')
    print(f'✅ plotly  : {plotly.__version__}')
    print(f'✅ numpy   : {numpy.__version__}')
    print('모든 패키지가 설치되어 있습니다!')


✅ torch   : 2.11.0+cu130
✅ plotly  : 6.7.0
✅ numpy   : 2.4.4
모든 패키지가 설치되어 있습니다!


# 🌊 FNO로 풀어보는 Darcy Flow
## Fourier Neural Operator — 발표 자료

**Neural Operator**는 편미분방정식(PDE)의 해를 *함수 공간 사이의 매핑*으로 학습하는 새로운 패러다임입니다.
이 노트북에서는 **Fourier Neural Operator (FNO)** 를 사용하여 **Darcy Flow** 문제를 단계별로 살펴봅니다.

---
### 📌 목차
1. 🌊 문제 소개: Darcy Flow와 Neural Operator
2. 📦 환경 설정 및 데이터 로딩
3. 📊 데이터 탐색 (Interactive Plotly)
4. 🔬 FNO 아키텍처 이해
5. 🏋️ 모델 학습 (Training)
6. 🎯 추론 및 예측 시각화
7. 🚀 제로샷 초해상도 (Zero-shot Super-resolution)
8. 📈 주파수 스펙트럼 분석
9. 📋 요약 및 결론

## 1. 🌊 문제 소개: Darcy Flow와 Neural Operator

### 물리적 배경

**Darcy Flow** (다시 흐름) 는 다공성 매질(porous medium)을 통과하는 유체의 흐름을 기술하는
대표적인 2차 타원형 편미분방정식(PDE)입니다.

> 응용: 지하수 흐름, 석유/가스 채굴, 필터 설계, 탄소 격리(Carbon Sequestration) 등

### 수학적 공식

$$-\nabla \cdot (a(\mathbf{x})\, \nabla u(\mathbf{x})) = f(\mathbf{x}), \quad \mathbf{x} \in D \subset \mathbb{R}^2$$

| 기호 | 의미 | 역할 |
|------|------|------|
| $a(\mathbf{x})$ | **투수율(permeability) 필드** | **입력 (Input)** |
| $u(\mathbf{x})$ | **압력(pressure) 필드** | **출력 (Output)** |
| $f(\mathbf{x})$ | 소스 항 (source term) = 1 | 상수 |
| $\partial D$ | 영역 경계 (Dirichlet BC) | $u=0$ |

### 고전적 방법의 한계

| 방법 | 특징 | 문제점 |
|------|------|--------|
| FEM / FDM | 정확한 수치해 | 새 $a(x)$마다 처음부터 $O(N^3)$ |
| Monte Carlo | 불확실성 정량화 가능 | 수천 번의 PDE 풀기 필요 |
| ROM | 줄어든 차원 | 비선형 문제에 취약 |

### 해결책: Neural Operator

$$\mathcal{G}_\theta: a \mapsto u, \quad \mathcal{G}_\theta \approx \mathcal{G}^\dagger$$

Neural Operator는 **함수 공간 → 함수 공간**의 매핑을 학습합니다:
- ✅ 새 $a(x)$에 대해 **밀리초 내 추론**
- ✅ 훈련 해상도와 다른 해상도에서도 동작 (**Resolution Invariant**)
- ✅ 전통적 방법 대비 **1000배 이상** 빠른 예측


In [3]:
import torch
import numpy as np
import plotly.graph_objects as go
import plotly.express as px
from plotly.subplots import make_subplots
import warnings
warnings.filterwarnings('ignore')

# neuraloperator 라이브러리
from neuralop.models import FNO
from neuralop.training import AdamW
from neuralop.data.datasets import load_darcy_flow_small
from neuralop.utils import count_model_params
from neuralop import LpLoss, H1Loss
from neuralop.layers.embeddings import GridEmbedding2D

device = torch.device('cuda' if torch.cuda.is_available() else 'cpu')
print(f'✅ 사용 장치: {device}')
print(f'✅ PyTorch 버전: {torch.__version__}')


✅ 사용 장치: cpu
✅ PyTorch 버전: 2.11.0+cu130


## 2. 📦 데이터 로딩

`load_darcy_flow_small` 함수로 소규모 Darcy Flow 데이터셋을 불러옵니다.
데이터는 처음 실행 시 자동으로 다운로드됩니다 (~수십 MB).

- **학습 해상도**: 16×16 (1,000 샘플)
- **테스트 해상도**: 16×16 (100 샘플), 32×32 (50 샘플)
- **입력**: 투수율 필드 $a(x)$ — 불연속 이진(binary) 패턴
- **출력**: 압력 필드 $u(x)$ — 부드러운 연속 함수

In [4]:
print('📥 Darcy Flow 데이터셋 로딩 중...')

train_loader, test_loaders, data_processor = load_darcy_flow_small(
    n_train=1000,
    batch_size=64,
    n_tests=[100, 50],
    test_resolutions=[16, 32],
    test_batch_sizes=[32, 32],
)
data_processor = data_processor.to(device)

print(f'✅ 학습 데이터  : {len(train_loader.dataset):,} 샘플')
print(f'✅ 테스트 (16×16): {len(test_loaders[16].dataset):,} 샘플')
print(f'✅ 테스트 (32×32): {len(test_loaders[32].dataset):,} 샘플')

# 데이터 형상 확인
batch = next(iter(train_loader))
print(f'\n📐 입력 (투수율 a) 형상: {batch["x"].shape}  → (배치, 채널, H, W)')
print(f'📐 출력 (압력  u) 형상: {batch["y"].shape}  → (배치, 채널, H, W)')


📥 Darcy Flow 데이터셋 로딩 중...
Loading test db for resolution 16 with 100 samples 
Loading test db for resolution 32 with 50 samples 
✅ 학습 데이터  : 1,000 샘플
✅ 테스트 (16×16): 50 샘플
✅ 테스트 (32×32): 50 샘플


/home/hson/work/neuraloperator/.venv/lib/python3.14/site-packages/torch/utils/data/dataloader.py:1118: UserWarning: 'pin_memory' argument is set as true but no accelerator is found, then device pinned memory won't be used.
  super().__init__(loader)



📐 입력 (투수율 a) 형상: torch.Size([64, 1, 16, 16])  → (배치, 채널, H, W)
📐 출력 (압력  u) 형상: torch.Size([64, 1, 16, 16])  → (배치, 채널, H, W)


## 3. 📊 데이터 탐색

슬라이더로 다양한 샘플을 탐색해보세요.
- **왼쪽**: 투수율 $a(x)$ — 이진(0/1) 패턴 (파란=낮음, 빨간=높음)
- **오른쪽**: 압력 $u(x)$ — 연속 스칼라장 (보라→노랑)

In [5]:
# ── 여러 샘플을 Interactive 슬라이더로 탐색 ──
test_samples_16 = test_loaders[16].dataset
N_SHOW = 8  # 보여줄 샘플 수

perms, pressures = [], []
for i in range(N_SHOW):
    d = test_samples_16[i]
    perms.append(d['x'][0].numpy())       # permeability (16x16)
    pressures.append(d['y'][0].numpy())   # pressure    (16x16)

fig = make_subplots(
    rows=1, cols=2,
    subplot_titles=['입력: 투수율 a(x)', '출력: 압력 u(x)'],
    horizontal_spacing=0.12,
)

for i in range(N_SHOW):
    vis = (i == 0)
    fig.add_trace(
        go.Heatmap(z=perms[i], colorscale='RdBu_r', showscale=True,
                   colorbar=dict(x=0.44, title='a(x)', len=0.75),
                   visible=vis, name=f'perm_{i}'),
        row=1, col=1,
    )
    fig.add_trace(
        go.Heatmap(z=pressures[i], colorscale='Plasma', showscale=True,
                   colorbar=dict(x=1.02, title='u(x)', len=0.75),
                   visible=vis, name=f'pres_{i}'),
        row=1, col=2,
    )

steps = []
for i in range(N_SHOW):
    vis_mask = [False] * (2 * N_SHOW)
    vis_mask[2 * i]     = True
    vis_mask[2 * i + 1] = True
    steps.append(dict(
        method='update',
        args=[{'visible': vis_mask},
              {'title': f'Darcy Flow 샘플 {i+1} / {N_SHOW} — 슬라이더로 전환하세요'}],
        label=f'{i+1}',
    ))

fig.update_layout(
    sliders=[dict(
        active=0, steps=steps,
        currentvalue={'prefix': '샘플: ', 'font': {'size': 14}},
        pad={'t': 55, 'b': 10},
    )],
    title=f'Darcy Flow 샘플 1 / {N_SHOW} — 슬라이더로 전환하세요',
    height=420,
    template='plotly_white',
    margin=dict(t=80, b=80),
)
fig.show()


In [6]:
# ── 데이터 통계 요약 ──
all_perms = np.stack(perms)      # (N, 16, 16)
all_pres  = np.stack(pressures)  # (N, 16, 16)

fig = make_subplots(
    rows=1, cols=2,
    subplot_titles=['투수율 a(x) 분포 (픽셀 값)',
                    '압력 u(x) 분포 (픽셀 값)'],
)

fig.add_trace(
    go.Histogram(x=all_perms.flatten(), nbinsx=40,
                 marker_color='steelblue', opacity=0.8, name='a(x)'),
    row=1, col=1,
)
fig.add_trace(
    go.Histogram(x=all_pres.flatten(), nbinsx=60,
                 marker_color='tomato', opacity=0.8, name='u(x)'),
    row=1, col=2,
)

fig.update_layout(
    height=350, template='plotly_white',
    title='데이터 분포 — 투수율(이진 패턴) vs 압력(연속 분포)',
    showlegend=False,
)
fig.update_xaxes(title_text='픽셀 값', row=1, col=1)
fig.update_xaxes(title_text='픽셀 값', row=1, col=2)
fig.update_yaxes(title_text='빈도', row=1, col=1)
fig.show()

print(f'투수율 a(x): 고유값 = {np.unique(all_perms).round(3)}  (이진 패턴)')
print(f'압력  u(x): 범위 = [{all_pres.min():.4f}, {all_pres.max():.4f}]')


투수율 a(x): 고유값 = [0. 1.]  (이진 패턴)
압력  u(x): 범위 = [0.0000, 1.6146]


## 4. 🔬 FNO 아키텍처 이해

### 핵심 아이디어

**Fourier Neural Operator (FNO)** (Li et al., 2021) 는 스펙트럼 도메인에서 컨볼루션을 수행합니다.

$$(\mathcal{K}(a;\phi)\,v_t)(x) = \mathcal{F}^{-1}\bigl(R_\phi \cdot \mathcal{F}(v_t)\bigr)(x)$$

여기서:
- $\mathcal{F}$: 고속 푸리에 변환 (FFT)
- $R_\phi$: 저주파 모드에만 적용되는 학습 가능한 복소수 가중치
- $\mathcal{F}^{-1}$: 역 FFT

### 아키텍처 파이프라인

```
Input a(x)         Lifting          FNO Block × L         Projection       Output û(x)
(B, 1, H, W)  →  (B, 24, H, W)  →  (B, 24, H, W)  →   (B, 1, H, W)  →  (B, 1, H, W)
                   Channel MLP       Spectral Conv          Channel MLP
                                   + Linear Skip
                                   + Channel MLP
```

### 각 FNO 블록 내부

```
      v_t(x)
       │
       ├────[SpectralConv]──── FFT → 저주파 모드 n개만 유지 → R_φ 곱 → IFFT ──┐
       │                                                                       + → σ → v_{t+1}
       └────[Linear W]────────────────── 1×1 Convolution ─────────────────────┘
```

### FNO의 Resolution Invariance (해상도 불변성)

> 모델은 **연속 함수** 사이의 매핑을 학습하기 때문에,
> 훈련 해상도(16×16)와 다른 해상도(32×32, 64×64)에서도 동작합니다.


In [7]:
# ── FNO 아키텍처 다이어그램 (Plotly) ──

COLORS = dict(
    input='#1565C0',       # deep blue
    lifting='#2E7D32',     # deep green
    fno='#E65100',         # deep orange
    proj='#6A1B9A',        # deep purple
    output='#B71C1C',      # deep red
    arrow='#546E7A',
)

def box(fig, x0, y0, x1, y1, color, text, fs=11):
    fig.add_shape(type='rect', x0=x0, y0=y0, x1=x1, y1=y1,
                  fillcolor=color, opacity=0.85,
                  line=dict(color='white', width=2))
    fig.add_annotation(
        x=(x0+x1)/2, y=(y0+y1)/2, text=text,
        showarrow=False, font=dict(color='white', size=fs),
        align='center',
    )

def arrow(fig, x0, y0, x1, y1, color='#546E7A'):
    fig.add_annotation(
        x=x1, y=y1, ax=x0, ay=y0,
        xref='x', yref='y', axref='x', ayref='y',
        showarrow=True, arrowhead=3, arrowwidth=2.5,
        arrowcolor=color,
    )

fig = go.Figure()

# ── Main pipeline ──
xs = [0, 2.0, 3.9, 5.7, 7.5, 9.3, 11.1, 13.1, 14.9]
w  = 1.6

box(fig, xs[0], 0.25, xs[0]+w, 0.75, COLORS['input'],
    'Input<br><i>a(x)</i><br>(B,1,H,W)')

arrow(fig, xs[0]+w, 0.5, xs[1], 0.5)

box(fig, xs[1], 0.15, xs[1]+w, 0.85, COLORS['lifting'],
    'Lifting<br>MLP<br>1→24 ch')

arrow(fig, xs[1]+w, 0.5, xs[2], 0.5)

for k, i in enumerate([2, 3, 4, 5]):
    x0 = xs[i]
    box(fig, x0, 0.10, x0+w, 0.90, COLORS['fno'],
        f'FNO<br>Block {k+1}<br>24 ch', fs=10)
    if k < 3:
        arrow(fig, x0+w, 0.5, xs[i+1], 0.5)

arrow(fig, xs[5]+w, 0.5, xs[6], 0.5)

box(fig, xs[6], 0.15, xs[6]+w, 0.85, COLORS['proj'],
    'Projection<br>MLP<br>24→1 ch')

arrow(fig, xs[6]+w, 0.5, xs[7], 0.5)

box(fig, xs[7], 0.25, xs[7]+w, 0.75, COLORS['output'],
    'Output<br><i>û(x)</i><br>(B,1,H,W)')

# ── Detail: Inside an FNO Block ──
bx, by = 3.9, -1.6
bw, bh = 7.5, 1.2

fig.add_shape(type='rect', x0=bx, y0=by, x1=bx+bw, y1=by+bh,
              line=dict(color='#E65100', width=1.5, dash='dash'), fillcolor='#FFF3E0')
fig.add_annotation(x=bx+bw/2, y=by+bh+0.08, text='<b>FNO Block 내부 구조</b>',
                   showarrow=False, font=dict(color='#E65100', size=12))

# Spectral conv path
arrow(fig, bx+0.1, by+bh*0.7, bx+1.2, by+bh*0.7)
box(fig, bx+1.2, by+bh*0.5, bx+3.2, by+bh*0.95, '#EF6C00',
    'Spectral Conv<br>FFT→modes→R_φ→IFFT', fs=9)

# Linear skip path
arrow(fig, bx+0.1, by+bh*0.25, bx+1.2, by+bh*0.25)
box(fig, bx+1.2, by+0.05, bx+3.2, by+bh*0.45, '#5D4037',
    'Linear Skip<br>(1×1 Conv)', fs=9)

# Sum
arrow(fig, bx+3.2, by+bh*0.7, bx+4.0, by+bh*0.5)
arrow(fig, bx+3.2, by+bh*0.25, bx+4.0, by+bh*0.45)
fig.add_shape(type='circle', x0=bx+4.0, y0=by+bh*0.35, x1=bx+4.4, y1=by+bh*0.65,
              fillcolor='#795548', line=dict(color='white'))
fig.add_annotation(x=bx+4.2, y=by+bh*0.5, text='<b>+</b>',
                   showarrow=False, font=dict(color='white', size=14))

# GELU
arrow(fig, bx+4.4, by+bh*0.5, bx+5.1, by+bh*0.5)
box(fig, bx+5.1, by+bh*0.3, bx+6.3, by+bh*0.7, '#388E3C', 'σ<br>GELU', fs=11)
arrow(fig, bx+6.3, by+bh*0.5, bx+7.2, by+bh*0.5)

# Input/output labels
fig.add_annotation(x=bx+0.1, y=by+bh*0.5, text='v_t<br>in',
                   showarrow=False, font=dict(color='#E65100', size=9), xanchor='right')
fig.add_annotation(x=bx+7.3, y=by+bh*0.5, text='v_{t+1}<br>out',
                   showarrow=False, font=dict(color='#E65100', size=9))

# Connect main diagram to detail
fig.add_shape(type='line', x0=5.7+0.8, y0=0.10, x1=5.7+0.8, y1=-0.4,
              line=dict(color='#E65100', dash='dot', width=1))
fig.add_shape(type='line', x0=5.7+0.8, y0=-0.4, x1=bx+bw/2, y1=-0.4,
              line=dict(color='#E65100', dash='dot', width=1))
arrow(fig, bx+bw/2, -0.4, bx+bw/2, by+bh+0.01, color='#E65100')

fig.update_layout(
    xaxis=dict(range=[-0.3, 17], showticklabels=False, showgrid=False, zeroline=False),
    yaxis=dict(range=[-1.8, 1.3], showticklabels=False, showgrid=False, zeroline=False),
    height=420, margin=dict(t=40, b=10, l=10, r=10),
    title='FNO 아키텍처 다이어그램 (위: 전체 파이프라인, 아래: FNO 블록 내부)',
    template='plotly_white',
    plot_bgcolor='#FAFAFA',
)
fig.show()


In [23]:
# ── Spectral Convolution 시각화 ──
# FFT로 투수율 필드를 주파수 도메인으로 변환하고,
# FNO가 사용하는 저주파 모드만 유지했을 때 어떻게 되는지 시각화

sample = test_samples_16[0]
a_field = np.squeeze(np.asarray(sample['x'][0].cpu()))
if a_field.ndim != 2:
    a_field = a_field.reshape(a_field.shape[-2], a_field.shape[-1])

# 2D FFT
A_fft = np.fft.fft2(a_field)
A_mag = np.abs(np.fft.fftshift(A_fft))

# 모드 트런케이션 시뮬레이션
results = {}
for n_modes in [2, 4, 8]:
    mask = np.zeros_like(A_fft)
    mask[:n_modes, :n_modes] = 1
    mask[-n_modes:, :n_modes] = 1
    mask[:n_modes, -n_modes:] = 1
    mask[-n_modes:, -n_modes:] = 1
    A_filtered = A_fft * mask
    results[n_modes] = np.real(np.fft.ifft2(A_filtered))

# 2×4에서 빈 칸용 Scatter 스페이서는 빈 패널처럼 보이므로 2×3으로 배치
# 3열×2행은 도메인 기본 종횡비가 가로로 길어지므로, figure 크기 비율을 cols/rows로 맞춘다.
_GRID_ROWS, _GRID_COLS = 2, 3
_FIG_H_PX = 540

fig = make_subplots(
    rows=_GRID_ROWS,
    cols=_GRID_COLS,
    subplot_titles=[
        '원본 a(x)',
        '푸리에 스펙트럼<br>|F[a]| (log scale)',
        '복원: 2 modes<br>(FNO 초저해상도)',
        '복원: 4 modes',
        '복원: 8 modes<br>(FNO 기본값)',
        '원본 vs 8-mode 오차',
    ],
    horizontal_spacing=0.05,
    vertical_spacing=0.12,
)

cmap_field = 'RdBu_r'
cmap_spec = 'Hot'

fig.add_trace(go.Heatmap(z=a_field, colorscale=cmap_field, showscale=False), row=1, col=1)
fig.add_trace(go.Heatmap(z=np.log1p(A_mag), colorscale=cmap_spec, showscale=False), row=1, col=2)
fig.add_trace(go.Heatmap(z=results[2], colorscale=cmap_field, showscale=False), row=1, col=3)
fig.add_trace(go.Heatmap(z=results[4], colorscale=cmap_field, showscale=False), row=2, col=1)
fig.add_trace(go.Heatmap(z=results[8], colorscale=cmap_field, showscale=False), row=2, col=2)
error_8 = np.abs(a_field - results[8])
fig.add_trace(go.Heatmap(z=error_8, colorscale='Reds', showscale=False), row=2, col=3)

fig.update_layout(
    height=_FIG_H_PX,
    width=int(_FIG_H_PX * _GRID_COLS / _GRID_ROWS),
    template='plotly_white',
    title='Spectral Convolution 시각화: 저주파 모드(n_modes)만 유지하면?',
    margin=dict(t=80, b=20, l=10, r=10),
    showlegend=False,
)
fig.update_xaxes(showticklabels=False)
fig.update_yaxes(showticklabels=False)
# 데이터 좌표도 1:1 (16×16과 함께 쓰면 셀 모양이 더 안정적)
for row in range(1, _GRID_ROWS + 1):
    for col in range(1, _GRID_COLS + 1):
        fig.update_yaxes(scaleanchor='x', scaleratio=1, row=row, col=col)
fig.show()

print('💡 핵심: FNO는 n_modes=8 개의 저주파 모드만 사용하여 학습합니다.')
print('   고주파 노이즈를 무시하고 전역적인 패턴을 포착합니다.')


💡 핵심: FNO는 n_modes=8 개의 저주파 모드만 사용하여 학습합니다.
   고주파 노이즈를 무시하고 전역적인 패턴을 포착합니다.


## 5. 🏋️ 모델 생성 및 학습

### FNO 모델 파라미터 설명

| 파라미터 | 값 | 의미 |
|---------|---|------|
| `n_modes` | (8, 8) | 유지할 푸리에 모드 수 (x, y 방향) |
| `in_channels` | 1 | 입력 채널 수 (투수율 1개) |
| `out_channels` | 1 | 출력 채널 수 (압력 1개) |
| `hidden_channels` | 24 | 잠재 표현의 채널 폭 |
| `n_layers` | 4 | FNO 블록 수 (기본값) |
| `projection_channel_ratio` | 2 | 투영 채널 비율 |


In [9]:
# ── FNO 모델 생성 ──
model = FNO(
    n_modes=(8, 8),
    in_channels=1,
    out_channels=1,
    hidden_channels=24,
    projection_channel_ratio=2,
)
model = model.to(device)

n_params = count_model_params(model)
print(f'✅ 모델 파라미터 수: {n_params:,}')
print()

# 레이어별 파라미터 수 시각화
layer_names, layer_params = [], []
for name, param in model.named_parameters():
    if param.requires_grad:
        layer_names.append(name)
        layer_params.append(param.numel())

# 상위 15개 레이어
top_k = 15
sorted_idx = sorted(range(len(layer_params)), key=lambda i: layer_params[i], reverse=True)
top_names  = [layer_names[i].replace('fno_blocks.', '').replace('weight', 'W')[:40]
              for i in sorted_idx[:top_k]]
top_params = [layer_params[i] for i in sorted_idx[:top_k]]

fig = go.Figure(go.Bar(
    x=top_params[::-1],
    y=top_names[::-1],
    orientation='h',
    marker_color='#1976D2',
    text=[f'{p:,}' for p in top_params[::-1]],
    textposition='outside',
))
fig.update_layout(
    title=f'FNO 레이어별 파라미터 수 (총 {n_params:,}개) — Top {top_k}',
    xaxis_title='파라미터 수',
    height=480,
    template='plotly_white',
    margin=dict(l=260, r=60),
)
fig.show()


✅ 모델 파라미터 수: 191,881



In [10]:
# ── 학습 설정 ──
optimizer = AdamW(model.parameters(), lr=1e-2, weight_decay=1e-4)
scheduler = torch.optim.lr_scheduler.CosineAnnealingLR(optimizer, T_max=20)

l2loss    = LpLoss(d=2, p=2)   # L2: 함수값 오차
h1loss    = H1Loss(d=2)        # H1: 함수값 + 그래디언트 오차
train_loss = h1loss

print('✅ 옵티마이저: AdamW (lr=1e-2, weight_decay=1e-4)')
print('✅ 스케줄러  : CosineAnnealing (T_max=20)')
print('✅ 학습 손실 : H1Loss (함수값 + 그래디언트 포함)')
print('✅ 평가 손실 : L2Loss, H1Loss')


✅ 옵티마이저: AdamW (lr=1e-2, weight_decay=1e-4)
✅ 스케줄러  : CosineAnnealing (T_max=20)
✅ 학습 손실 : H1Loss (함수값 + 그래디언트 포함)
✅ 평가 손실 : L2Loss, H1Loss


In [11]:
# ── 학습 루프 ──
N_EPOCHS = 20

train_losses   = []
val_losses_h1  = []
val_losses_l2  = []

print(f'{'에포크':>6} | {'학습(H1)':>10} | {'검증(H1)':>10} | {'검증(L2)':>10}')
print('-' * 46)

for epoch in range(N_EPOCHS):
    # ── Training ──
    model.train()
    ep_loss, n_batches = 0.0, 0
    for batch in train_loader:
        batch = data_processor.preprocess(batch)
        x = batch['x'].to(device)
        y = batch['y'].to(device)
        optimizer.zero_grad()
        pred = model(x)
        loss = train_loss(pred.float(), y.float())
        loss.backward()
        optimizer.step()
        ep_loss += loss.item()
        n_batches += 1
    avg_train = ep_loss / n_batches
    train_losses.append(avg_train)

    # ── Validation ──
    model.eval()
    vh1, vl2, nv = 0.0, 0.0, 0
    with torch.no_grad():
        for batch in test_loaders[16]:
            batch = data_processor.preprocess(batch)
            x = batch['x'].to(device)
            y = batch['y'].to(device)
            pred = model(x)
            vh1 += h1loss(pred.float(), y.float()).item()
            vl2 += l2loss(pred.float(), y.float()).item()
            nv  += 1
    val_losses_h1.append(vh1 / nv)
    val_losses_l2.append(vl2 / nv)

    scheduler.step()

    if (epoch + 1) % 5 == 0 or epoch == 0:
        print(f'{epoch+1:>6} | {train_losses[-1]:>10.5f} | '
              f'{val_losses_h1[-1]:>10.5f} | {val_losses_l2[-1]:>10.5f}')

print()
print(f'✅ 학습 완료!  최종 검증 L2: {val_losses_l2[-1]:.4f}')


   에포크 |     학습(H1) |     검증(H1) |     검증(L2)
----------------------------------------------


/home/hson/work/neuraloperator/.venv/lib/python3.14/site-packages/torch/utils/data/dataloader.py:1118: UserWarning: 'pin_memory' argument is set as true but no accelerator is found, then device pinned memory won't be used.
  super().__init__(loader)


     1 |   50.17752 |   15.59155 |   15.06825
     5 |   17.61387 |    6.94057 |    7.26564
    10 |   12.84218 |    5.33629 |    5.35326
    15 |   11.56659 |    4.98276 |    4.81399
    20 |   11.04612 |    4.84675 |    4.55800

✅ 학습 완료!  최종 검증 L2: 4.5580


In [12]:
# ── 학습 곡선 시각화 ──
epochs = list(range(1, N_EPOCHS + 1))

fig = make_subplots(
    rows=1, cols=2,
    subplot_titles=['학습 손실 (H1Loss)', '검증 손실 비교'],
    horizontal_spacing=0.1,
)

# Train H1
fig.add_trace(go.Scatter(
    x=epochs, y=train_losses,
    mode='lines+markers', name='학습 H1',
    line=dict(color='#1565C0', width=2.5),
    marker=dict(size=5),
), row=1, col=1)

# Val H1
fig.add_trace(go.Scatter(
    x=epochs, y=val_losses_h1,
    mode='lines+markers', name='검증 H1',
    line=dict(color='#E65100', width=2.5),
    marker=dict(size=5),
), row=1, col=2)

# Val L2
fig.add_trace(go.Scatter(
    x=epochs, y=val_losses_l2,
    mode='lines+markers', name='검증 L2',
    line=dict(color='#2E7D32', width=2.5, dash='dash'),
    marker=dict(size=5),
), row=1, col=2)

fig.update_xaxes(title_text='에포크')
fig.update_yaxes(title_text='손실값', type='log')
fig.update_layout(
    height=380,
    template='plotly_white',
    title='FNO 학습 곡선',
    legend=dict(x=0.52, y=0.95),
)
fig.show()

best_l2 = min(val_losses_l2)
best_ep = val_losses_l2.index(best_l2) + 1
print(f'최저 검증 L2: {best_l2:.4f}  (에포크 {best_ep})')


최저 검증 L2: 4.5580  (에포크 20)


## 6. 🎯 추론 및 예측 시각화

학습된 FNO 모델로 테스트 샘플을 예측하고 결과를 비교합니다.

- **왼쪽**: 입력 투수율 $a(x)$ (원본)
- **가운데**: 정답 압력 $u(x)$ (수치해)
- **오른쪽**: FNO 예측 $\hat{u}(x)$

> 정답과 예측이 얼마나 일치하는지 확인하세요!

In [25]:
# ── 추론: 3개 샘플 비교 (입력 / 정답 / 예측) ──
model.eval()
N_COMPARE = 3
_GRID_ROWS, _GRID_COLS = N_COMPARE, 3
# 패널당 목표 픽셀(대략 정사각) + 좌우 컬러바용 마진 — 값만 키우면 전체가 넉넉해짐
_CELL_PX = 320
_GRID_PAD_X, _GRID_PAD_Y = 1.14, 1.18
_BODY_W = _CELL_PX * _GRID_COLS * _GRID_PAD_X
_BODY_H = _CELL_PX * _GRID_ROWS * _GRID_PAD_Y
_MARGIN_L, _MARGIN_R = 118, 118
_MARGIN_T, _MARGIN_B = 105, 78

fig = make_subplots(
    rows=_GRID_ROWS,
    cols=_GRID_COLS,
    subplot_titles=(
        ['입력 a(x)', '정답 u(x)', '예측 û(x)'] * N_COMPARE
    ),
    horizontal_spacing=0.07,
    vertical_spacing=0.09,
)

l2_errors = []

with torch.no_grad():
    for idx in range(N_COMPARE):
        sample     = test_samples_16[idx]
        sample_proc = data_processor.preprocess(sample, batched=False)
        x_proc = sample_proc['x'].to(device)
        y_true = sample_proc['y']

        pred = model(x_proc.unsqueeze(0)).squeeze(0).cpu()

        perm = np.asarray(sample['x'][0].cpu())
        gt = np.asarray(y_true.squeeze().detach().cpu())
        pr = np.asarray(pred.squeeze().detach().cpu())
        if gt.ndim != 2:
            gt = gt.reshape(gt.shape[-2], gt.shape[-1])
        if pr.ndim != 2:
            pr = pr.reshape(pr.shape[-2], pr.shape[-1])

        # L2 relative error
        err = np.linalg.norm(pr - gt) / (np.linalg.norm(gt) + 1e-8)
        l2_errors.append(err)

        row = idx + 1
        vmin = float(min(gt.min(), pr.min()))
        vmax = float(max(gt.max(), pr.max()))

        _mid_row = N_COMPARE // 2
        cb_a = dict(
            thickness=18,
            len=0.52,
            title='a(x)',
            x=-0.04,
            xanchor='right',
            outlinewidth=0,
            tickfont=dict(size=11),
        )
        cb_u = dict(
            thickness=18,
            len=0.52,
            title='u(x)',
            x=1.06,
            xanchor='left',
            outlinewidth=0,
            tickfont=dict(size=11),
        )

        # 행마다 z 범위가 다르므로 layout의 단일 coloraxis를 여러 Heatmap과 공유하면
        # Plotly에서 일부 패널(정답 열 등)이 비어 보일 수 있음 → 패널별 colorscale 사용.
        fig.add_trace(
            go.Heatmap(
                z=perm,
                colorscale='RdBu_r',
                showscale=(idx == _mid_row),
                colorbar=cb_a,
            ),
            row=row, col=1,
        )
        fig.add_trace(
            go.Heatmap(z=gt, colorscale='Plasma', showscale=False,
                       zmin=vmin, zmax=vmax),
            row=row, col=2,
        )
        fig.add_trace(
            go.Heatmap(
                z=pr,
                colorscale='Plasma',
                showscale=(idx == _mid_row),
                zmin=vmin,
                zmax=vmax,
                colorbar=cb_u,
            ),
            row=row, col=3,
        )

        # L2 error annotation on prediction panel (축 도메인 아래로 빼서 히트맵과 분리)
        fig.add_annotation(
            x=0.5, y=-0.22, xref=f'x{(idx * 3 + 3)}', yref=f'y{(idx * 3 + 3)}',
            text=f'L2 오차: {err:.4f}',
            showarrow=False, font=dict(size=11, color='red'),
            xanchor='center', yanchor='top',
        )

fig.update_layout(
    width=int(_BODY_W + _MARGIN_L + _MARGIN_R),
    height=int(_BODY_H + _MARGIN_T + _MARGIN_B),
    template='plotly_white',
    title=dict(
        text=f'FNO 예측 결과 비교 (평균 L2 오차: {np.mean(l2_errors):.4f})',
        x=0.5,
        xanchor='center',
    ),
    margin=dict(l=_MARGIN_L, r=_MARGIN_R, t=_MARGIN_T, b=_MARGIN_B),
)
fig.update_xaxes(showticklabels=False)
fig.update_yaxes(showticklabels=False)
for row in range(1, _GRID_ROWS + 1):
    for col in range(1, _GRID_COLS + 1):
        fig.update_yaxes(scaleanchor='x', scaleratio=1, row=row, col=col)
fig.show()

for i, e in enumerate(l2_errors):
    print(f'  샘플 {i+1}: L2 상대 오차 = {e:.4f}')


  샘플 1: L2 상대 오차 = 0.1393
  샘플 2: L2 상대 오차 = 0.1139
  샘플 3: L2 상대 오차 = 0.1248


In [19]:
# ── 오차 분석: 절대 오차 히트맵 + 오차 분포 ──
model.eval()
N_ERR = 5

fig = make_subplots(
    rows=2, cols=N_ERR,
    subplot_titles=(
        [f'|오차| 샘플 {i+1}' for i in range(N_ERR)] +
        ['오차 분포 (전체 테스트셋)', '', '', '', '']
    ),
    horizontal_spacing=0.03,
    vertical_spacing=0.12,
    specs=[[{}]*N_ERR,
           [{'colspan': N_ERR}, None, None, None, None]],
)

all_errors = []

with torch.no_grad():
    for idx in range(N_ERR):
        sample      = test_samples_16[idx]
        sample_proc = data_processor.preprocess(sample, batched=False)
        x_proc = sample_proc['x'].to(device)
        pred_t = model(x_proc.unsqueeze(0)).squeeze(0).detach().cpu()
        y_true = np.asarray(sample_proc['y'].squeeze().detach().cpu())
        pred = np.asarray(pred_t.squeeze())
        if y_true.ndim != 2:
            y_true = y_true.reshape(y_true.shape[-2], y_true.shape[-1])
        if pred.ndim != 2:
            pred = pred.reshape(pred.shape[-2], pred.shape[-1])
        abs_err = np.abs(pred - y_true)

        fig.add_trace(
            go.Heatmap(z=abs_err, colorscale='Reds', showscale=(idx == N_ERR-1),
                       colorbar=dict(x=1.02, len=0.45, title='|오차|', thickness=12)),
            row=1, col=idx+1,
        )

    # 전체 테스트셋 L2 오차 수집
    all_l2 = []
    for batch in test_loaders[16]:
        batch = data_processor.preprocess(batch)
        x = batch['x'].to(device)
        y = batch['y'].to(device)
        pred = model(x)
        for b in range(pred.shape[0]):
            e = (torch.norm(pred[b] - y[b]) / (torch.norm(y[b]) + 1e-8)).item()
            all_l2.append(e)

fig.add_trace(
    go.Histogram(x=all_l2, nbinsx=30,
                 marker_color='#1565C0', opacity=0.8,
                 name='L2 오차'),
    row=2, col=1,
)
fig.add_vline(x=np.mean(all_l2), line_dash='dash', line_color='red',
              annotation_text=f'평균 {np.mean(all_l2):.4f}',
              annotation_position='top right', row=2, col=1)

fig.update_layout(
    height=520,
    template='plotly_white',
    title=f'오차 분석 — 평균 L2: {np.mean(all_l2):.4f}, 중앙값: {np.median(all_l2):.4f}',
    showlegend=False,
    margin=dict(r=60),
)
fig.update_xaxes(showticklabels=False, row=1)
fig.update_yaxes(showticklabels=False, row=1)
fig.update_xaxes(title_text='L2 상대 오차', row=2, col=1)
fig.update_yaxes(title_text='샘플 수', row=2, col=1)
fig.show()

print(f'전체 테스트셋 통계 ({len(all_l2)} 샘플):')
print(f'  평균 L2 오차: {np.mean(all_l2):.4f}')
print(f'  중앙값 L2 오차: {np.median(all_l2):.4f}')
print(f'  최대 L2 오차: {np.max(all_l2):.4f}')


/home/hson/work/neuraloperator/.venv/lib/python3.14/site-packages/torch/utils/data/dataloader.py:1118: UserWarning: 'pin_memory' argument is set as true but no accelerator is found, then device pinned memory won't be used.
  super().__init__(loader)


전체 테스트셋 통계 (50 샘플):
  평균 L2 오차: 0.1823
  중앙값 L2 오차: 0.1698
  최대 L2 오차: 0.3132


## 7. 🚀 제로샷 초해상도 (Zero-shot Super-resolution)

FNO의 가장 강력한 특징 중 하나: **재학습 없이** 더 높은 해상도로 예측 가능!

- **학습**: 16×16 해상도만 사용
- **테스트**: 32×32 해상도 입력 → FNO가 32×32 출력 생성

> 이것이 가능한 이유는 FNO가 **연속 함수 공간의 매핑**을 학습하기 때문입니다.
> 어떤 이산화(discretization)에도 의존하지 않습니다.

In [20]:
# ── 제로샷 초해상도 시각화 ──
model.eval()
test_samples_32 = test_loaders[32].dataset
N_SR = 3

fig = make_subplots(
    rows=N_SR, cols=4,
    subplot_titles=(
        ['16×16 입력', '16×16 정답', '32×32 입력', '32×32 FNO 예측'] * N_SR
    ),
    horizontal_spacing=0.04,
    vertical_spacing=0.06,
)

with torch.no_grad():
    for idx in range(N_SR):
        # 16x16 sample
        s16      = test_samples_16[idx]
        s16_proc = data_processor.preprocess(s16, batched=False)
        x16 = s16_proc['x'].to(device)
        p16_t = model(x16.unsqueeze(0)).squeeze(0).detach().cpu()
        y16 = np.asarray(s16_proc['y'].squeeze().detach().cpu())
        p16 = np.asarray(p16_t.squeeze())
        if y16.ndim != 2:
            y16 = y16.reshape(y16.shape[-2], y16.shape[-1])
        if p16.ndim != 2:
            p16 = p16.reshape(p16.shape[-2], p16.shape[-1])

        # 32x32 sample (same underlying function, higher resolution)
        s32      = test_samples_32[idx]
        s32_proc = data_processor.preprocess(s32, batched=False)
        x32 = s32_proc['x'].to(device)
        p32_t = model(x32.unsqueeze(0)).squeeze(0).detach().cpu()
        y32 = np.asarray(s32_proc['y'].squeeze().detach().cpu())
        p32 = np.asarray(p32_t.squeeze())
        if y32.ndim != 2:
            y32 = y32.reshape(y32.shape[-2], y32.shape[-1])
        if p32.ndim != 2:
            p32 = p32.reshape(p32.shape[-2], p32.shape[-1])

        row = idx + 1
        fig.add_trace(go.Heatmap(z=s16['x'][0].numpy(), colorscale='RdBu_r',
                                 showscale=False), row=row, col=1)
        fig.add_trace(go.Heatmap(z=y16, colorscale='Plasma',
                                 showscale=False), row=row, col=2)
        fig.add_trace(go.Heatmap(z=s32['x'][0].numpy(), colorscale='RdBu_r',
                                 showscale=False), row=row, col=3)
        fig.add_trace(go.Heatmap(z=p32, colorscale='Plasma',
                                 showscale=False), row=row, col=4)

        l2_16 = np.linalg.norm(p16 - y16) / (np.linalg.norm(y16) + 1e-8)
        l2_32 = np.linalg.norm(p32 - y32) / (np.linalg.norm(y32) + 1e-8)

        fig.add_annotation(
            x=0.5, y=-0.15, xref=f'x{idx*4+2}', yref=f'y{idx*4+2}',
            text=f'16×16 L2: {l2_16:.4f}', showarrow=False,
            font=dict(size=9, color='navy'), xanchor='center',
        )
        fig.add_annotation(
            x=0.5, y=-0.15, xref=f'x{idx*4+4}', yref=f'y{idx*4+4}',
            text=f'32×32 L2: {l2_32:.4f}', showarrow=False,
            font=dict(size=9, color='darkred'), xanchor='center',
        )

fig.update_layout(
    height=620,
    template='plotly_white',
    title='제로샷 초해상도: 16×16 학습 → 32×32 예측 (재학습 없음!)',
    margin=dict(t=80, b=30),
)
fig.update_xaxes(showticklabels=False)
fig.update_yaxes(showticklabels=False)
fig.show()

print('💡 핵심: 16×16으로만 학습했음에도 32×32에서 유사한 오차 수준을 달성!')
print('   이것이 Neural Operator의 Resolution Invariance입니다.')


💡 핵심: 16×16으로만 학습했음에도 32×32에서 유사한 오차 수준을 달성!
   이것이 Neural Operator의 Resolution Invariance입니다.


In [17]:
# ── 주파수 스펙트럼 분석 ──
# 투수율 a(x)와 압력 u(x)의 에너지 스펙트럼 비교

model.eval()
N_FFT = 20  # 스펙트럼 평균을 위한 샘플 수

spectra_a, spectra_u, spectra_pred = [], [], []

with torch.no_grad():
    for idx in range(N_FFT):
        sample      = test_samples_16[idx]
        sample_proc = data_processor.preprocess(sample, batched=False)
        x_proc = sample_proc['x'].to(device)
        y_true = sample_proc['y'][0].numpy()
        pred   = model(x_proc.unsqueeze(0)).squeeze(0).cpu()[0].numpy()

        a_raw = sample['x'][0].numpy()

        # 방사형(radial) 평균 파워 스펙트럼 계산
        def radial_psd(field):
            arr = np.asarray(field).squeeze()
            if arr.ndim == 3:
                arr = arr[0]
            elif arr.ndim != 2:
                raise ValueError(f'radial_psd: 2D 필드 필요, 실제 shape={np.asarray(field).shape}')
            F = np.fft.fftshift(np.fft.fft2(arr))
            power = np.abs(F) ** 2
            h, w = power.shape
            cy, cx = h // 2, w // 2
            Y, X = np.ogrid[:h, :w]
            R = np.sqrt((X - cx)**2 + (Y - cy)**2).astype(int)
            rmax = min(cy, cx)
            psd = np.array([power[R == r].mean() if (R == r).any() else 0
                            for r in range(rmax)])
            return psd

        spectra_a.append(radial_psd(a_raw))
        spectra_u.append(radial_psd(y_true))
        spectra_pred.append(radial_psd(pred))

# 평균 스펙트럼
psd_a    = np.mean(spectra_a,    axis=0)
psd_u    = np.mean(spectra_u,    axis=0)
psd_pred = np.mean(spectra_pred, axis=0)
freqs    = np.arange(len(psd_a))

fig = make_subplots(
    rows=1, cols=2,
    subplot_titles=['방사형 파워 스펙트럼 (로그 스케일)',
                    '2D 스펙트럼: a(x) vs u(x)'],
    horizontal_spacing=0.1,
)

fig.add_trace(go.Scatter(x=freqs, y=psd_a,    mode='lines',
                         name='투수율 a(x)',  line=dict(color='blue',  width=2.5)),
              row=1, col=1)
fig.add_trace(go.Scatter(x=freqs, y=psd_u,    mode='lines',
                         name='압력 u(x) 정답', line=dict(color='green', width=2.5)),
              row=1, col=1)
fig.add_trace(go.Scatter(x=freqs, y=psd_pred, mode='lines',
                         name='압력 û(x) 예측', line=dict(color='red',   width=2,
                                                          dash='dash')),
              row=1, col=1)

# FNO가 유지하는 모드 범위 강조
fig.add_vrect(x0=0, x1=8, fillcolor='orange', opacity=0.12,
              annotation_text='FNO 유지 모드 (n=8)',
              annotation_position='top left', row=1, col=1)

# 2D FFT 이미지
a_2dfft = np.log1p(np.abs(np.fft.fftshift(np.fft.fft2(
    test_samples_16[0]['x'][0].numpy()))))
_u_fft = np.asarray(
    data_processor.preprocess(test_samples_16[0], batched=False)['y'][0]).squeeze()
if _u_fft.ndim == 3:
    _u_fft = _u_fft[0]
u_2dfft = np.log1p(np.abs(np.fft.fftshift(np.fft.fft2(_u_fft))))

fig.add_trace(go.Heatmap(z=a_2dfft, colorscale='Hot', showscale=False,
                         name='|F[a]|'), row=1, col=2)

fig.update_xaxes(title_text='주파수 (모드 번호)', row=1, col=1)
fig.update_yaxes(title_text='파워 (log)', type='log', row=1, col=1)
fig.update_xaxes(showticklabels=False, row=1, col=2)
fig.update_yaxes(showticklabels=False, row=1, col=2)
fig.update_layout(
    height=400, template='plotly_white',
    title='주파수 스펙트럼 분석 — FNO가 중요한 저주파 성분을 잘 포착',
    legend=dict(x=0.45, y=0.95),
)
fig.show()

print('💡 투수율 a(x): 고주파 성분이 풍부 (이진 패턴)')
print('   압력 u(x): 저주파 성분 위주 (부드러운 분포)')
print('   → FNO의 n_modes=8 저주파 모드만으로 압력장을 잘 포착할 수 있음!')


💡 투수율 a(x): 고주파 성분이 풍부 (이진 패턴)
   압력 u(x): 저주파 성분 위주 (부드러운 분포)
   → FNO의 n_modes=8 저주파 모드만으로 압력장을 잘 포착할 수 있음!


## 9. 📋 요약 및 결론

### 핵심 정리

| 항목 | 내용 |
|------|------|
| **문제** | Darcy Flow: $-\nabla\cdot(a\nabla u)=f$ |
| **입력** | 투수율 필드 $a(x)$ (이진 패턴) |
| **출력** | 압력 필드 $u(x)$ (연속 스칼라장) |
| **모델** | FNO — 푸리에 스펙트럼 도메인 컨볼루션 |
| **파라미터** | ~수만 개 (소규모 버전) |
| **학습 해상도** | 16×16 |
| **Zero-shot 테스트** | 32×32 (재학습 없음) |

### FNO의 3대 장점

1. **🚀 빠른 추론**: 새 입력에 대해 밀리초 내 예측 (FEM 대비 1000배 이상)
2. **📐 해상도 불변**: 훈련 해상도 외 임의 해상도에서 즉시 동작
3. **📊 함수 학습**: 유한 차원 벡터가 아닌 연속 함수 공간 매핑

### 다음 단계

- **더 큰 데이터셋** / 높은 해상도 (128×128, 421×421) 사용
- **TFNO** (Tucker FNO): 같은 성능, 90% 파라미터 절감
- **3D 확장**: Navier-Stokes, 대기 모델링
- **SFNO**: 구면 도메인 (기상 예측)
- **GINO/FNOGNO**: 불규칙 격자 (CAD 형상, 메쉬)

---

### 참고 문헌

- Li et al. (2021). **Fourier Neural Operator for Parametric Partial Differential Equations.** ICLR 2021. [arXiv:2010.08895](https://arxiv.org/abs/2010.08895)
- Kovachki et al. (2021). **Neural Operator: Learning Maps Between Function Spaces.** JMLR 2023.
- NeuralOperator Library: [github.com/neuraloperator/neuraloperator](https://github.com/neuraloperator/neuraloperator)
